**Программа генерирует АВТОМАТИЧЕСКИЙ УКАЗАТЕЛЬ к какому-либо русскому переводу древнеиндийского текста. Для старта программы запустить первую ячейку.**

**ВНИМАНИЕ!** Перед запуском этой программы необходимо получить файл с морфосинтаксическим разбором текста - *выполнить deeppavlov_parsing.ipynb*.

In [14]:
%%time

#ввести название файла с текстом для обработки
file = str(input('Введите название текстового файла (н-р, Рамаяна_3.txt): '))

#ввести путь к папке с файлами на гугл-диске
path = str(input('Укажите путь к папке с файлами (н-р, /content/drive/My Drive/Colab Notebooks/диплом/): '))

#подключение к гугл-диску
from google.colab import drive
drive.mount('/content/drive')

!pip install pymorphy2

#импортирование библиотек
import re
import nltk
nltk.download('punkt')
import pymorphy2
morph = pymorphy2.MorphAnalyzer()
vs = ['а', 'я', 'ю', 'е', 'о', 'ы', 'и', 'у']

#выполнение функций программы
sans, sans_dict2, forn, rus_words, phrase, repl, ones, opts, index3, rus_index, tr, rusforms = open_files()
rus_index = decline(rus_index)
indecl, decl_d, stem_d, exep, hyph, phrase, rest = group_sans(index3, sans, sans_dict2, phrase, rus_index)
found, upper_words = search(file, rus_index, phrase, rest, hyph, rus_words, indecl, decl_d, stem_d, exep)
res = index_transform(found, upper_words, rusforms, sans, index3, tr, exep)
clean, for_dpavlov = unite2()
clean = depppavlov_proc(file, for_dpavlov, clean)
lexemes = index_unite(clean)


united = get_index(lexemes, file)
print('Автоматический указатель записан в файл automated_index_{}'.format(file))
get_index_forms(united, file, phrase, repl, rus_words, rus_index)
print('Рубрики указателя во всех встречающихся формах записаны в файл automated_index_forms_{}'.format(file))

Введите название текстового файла (н-р, Рамаяна_3.txt): 01_part.txt
Укажите путь к папке с файлами (н-р, /content/drive/My Drive/Colab Notebooks/диплом/): /content/drive/My Drive/Colab Notebooks/диплом/
Mounted at /content/drive
     |████████████████████████████████| 61kB 6.5MB/s 
     |████████████████████████████████| 8.2MB 12.7MB/s 
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
Объем списка Серенсена - 9460 слов
Объем словаря иностранных слов - 14687 слов
Объем корпуса русских слов - 2914651 слов
Объем списка санскритских слов - 352592 слов
Обработано 1 предложений из 32
Обработано 2 предложений из 32
Обработано 3 предложений из 32
Обработано 4 предложений из 32
Обработано 5 предложений из 32
Обработано 6 предложений из 32
Обработано 7 предложений из 32
Обработано 8 предложений из 32
Обработано 9 предложений из 32
Обработано 10 предложений из 32
Обработано 11 предложений из 32
Обработано 12 предложений из 32
Обработано 13 

Открываем нужные файлы

In [1]:
def open_files():
  #1. список Серенсена
  with open(path + '9460-osnov-sanskritskikh-slov.txt', 'r', encoding = 'utf-8') as r:
      lines = r.readlines()
  sans = [x.lower() for x in lines]
  sans = [re.sub('\n', '', x) for x in sans]
  print('Объем списка Серенсена - ' + str(len(sans)) + ' слов')


  #2. список санскритизмов из 354 тыс. слов
  with open(path + '384000.txt', 'r', encoding = 'utf-8') as n:
      sans_dict = n.readlines()
  sans_dict = [re.sub('\n','',x) for x in sans_dict]
  sans_dict = set(sans_dict)
  ###удаляем санскритизмы длиной 1-2 буквы
  sans_dict = [x for x in sans_dict if len(x) > 2]

  #3. словарь иностранных слов
  with open(path + 'foreign_words.txt', 'r', encoding = 'utf-8') as n:
      d = n.readlines()
  forn = [re.sub('\n', '', x) for x in d]
  forn = [x for x in forn if len(x) > 1]
  ###добавляем к списку иностранных слов свои исключения, чтобы удалить их из корпуса русских слов (чтобы корпус русских слов не отфильтровывал нужные санскритизмы, например, преты)
  for i in ['анил', 'анупа', 'ануп', 'крошу', 'ванг', 'прет']:
    forn.append(i)
  print('Объем словаря иностранных слов - ' + str(len(forn)) + ' слов')

  #4. корпус русских слов
  with open(path + 'dict.opcorpora.txt', 'r', encoding = 'utf-8') as n:
      rus_words = []
      lem_rus_words = []
      d = n.read()
      d = re.sub('\n', ' ', d)
      d = re.sub('\t', ' ', d)
      d = re.sub('(^|\s)[0-9]+', '||', d)
      parads = re.findall('\|\|[^\|]+', d)
      forn_sans = set(forn) | set(sans)
      for par in parads:
        try:
          lem = re.search('\|\|\s([А-Яа-яёЁ]+)', par).group(1)
          if 'Ё' in lem:
              lem = re.sub('Ё', 'Е', lem)
              
          if lem.lower() not in forn_sans:
              lem_rus_words.append(lem.lower())
              words = re.findall('[А-Яа-яЁё]+', par)
              for w in words:
                  if 'Ё' in w:
                      w = re.sub('Ё', 'Е', w)
                  rus_words.append(w.lower())
        except:
          continue
  rus_words.remove('даму')
  rus_words.remove('дама')
  rus_words.remove('кишку')
  rus_words.remove('пилу')
  rus_words.remove('руру')
  rus_words.remove('турья')
  rus_words.remove('турье')
  rus_words.remove('кшатрия')
  rus_words.remove('кшатрии')
  rus_words.append('и')
  cases = ['nomn', 'gent', 'datv', 'accs', 'ablt', 'loct']
  for case in cases:
    list(rus_words).append(morph.parse('акт')[0].inflect({case}).word)
  rus_words = [x for x in rus_words if len(x) > 1]
  rus_words = set(rus_words)
  lem_rus_words = set(lem_rus_words)
  print('Объем корпуса русских слов - ' + str(len(rus_words)) + ' слов')

  ###очищаем список санскритских слов от русских лемм
  sans_dict2 = [z for z in sans_dict if z not in lem_rus_words]
  print('Объем списка санскритских слов - ' + str(len(sans_dict2)) + ' слов')

  phrase = {}
  repl = {}
  index3 = []
  opts = {}

  #5. список рубрик с аннтоациями и пояснениями, отобранные из бумажного указателя к 3-му тому Махабхараты вручную
  with open(path + '3_INDEX_phrases.txt', 'r', encoding='utf-8') as r:
    for i in r:
      s = i.split(': ')
      ph = s[0]
      syn = re.sub('\n', '', s[1])
      if ' ' in ph or '-' in ph:
        phrase[ph] =  syn
      else:
        repl[ph] =  syn

  #6. список однословных рубрик из бумажного указателя
  with open(path + '3_INDEX_oneword.txt', 'r', encoding='utf-8') as r:
    ones = r.readlines()
  ones = [re.sub('\n', '', x) for x in ones]

  #7. список рубрик с эпитетами
  with open(path + '3_INDEX_options.txt', 'r', encoding = 'utf-8') as r:
    for i in r:
      m = i.split(': ')
      s = m[0].lower()
      index3.append(s)
      val = m[1].split('; ')
      arr = []
      for v in val:
        r = v.split(' - ')
        l = re.sub('\n', '', r[1])
        arr.append((r[0].lower(), l.lower()))
      opts[s] = arr

  index3 += list(repl.keys()) + ones

  #8. рубрики с русскими словами из бумажного указателя

  with open(path + 'rus_index.txt', 'r', encoding='utf-8') as n:
    rus_index = n.readlines()
  rus_index = [re.sub('\n', '', x) for x in rus_index]

  #9. санскритизмы в плюралисе

  with open(path + 'pluralis_племена.txt', 'r', encoding = 'utf-8') as r:
    tr = r.readlines()

  tr = [re.sub('\n', '', x.lower()) for x in tr]

  10.

  with open(path + 'rusforms.txt', 'r', encoding = 'utf-8') as t:
      rusforms = t.readlines()
  rusforms = [re.sub('\n', '', x) for x in rusforms]

  return sans, sans_dict2, forn, rus_words, phrase, repl, ones, opts, index3, rus_index, tr, rusforms

Склонение русских слов из указателя

In [2]:
def decline(rus_index):

  rus_index_one = []
  for i in rus_index:
    if re.search('^[А-Яа-яёЁ]+$', i):
      rus_index_one.append(re.search('^[А-Яа-яёЁ]+$', i).group(0))
  rus_index_phrase = list(set(rus_index) - set(rus_index_one))

  rus_index = {}
  cases = ['nomn', 'gent', 'datv', 'accs', 'ablt', 'loct']
  for i in rus_index_one:
    for case in cases:
      if i not in rus_index:
        rus_index[i] = [morph.parse(i)[0].inflect({case}).word]
      else:
        rus_index[i].append(morph.parse(i)[0].inflect({case}).word)

  both = ['вселенский владыка', 'великая смоковница', 'тройственная вселенная', 'мудрецы-цари', 'семь святых мудрецов', 'поминальная жертва', 'великий полководец', 'великий индра', 'святые мудрецы', 'мудрецы-боги', 'божественные мудрецы', 'великий бог', 'восемь чаш', 'тридцать богов', 'демоны-змеи', 'мудрецы-брахманы', 'брахманы-мудрецы', 'третье небо', 'три мира', 'долг, польза, любовь', 'тридцать три бога', 'великий владыка', 'цари-мудрецы', 'пять великих элементов']
  last = ['не из чрева рожденный']
  except_last = ['старший брат гады', 'веда гандхарвов']
  tcsh = ['шествующий по небу', 'равно владеющий и правой, и левой рукой', 'смущающий душу', 'имеющий знаком быка', 'несущий знак']

  for i in rus_index_phrase:
    st = i.split()
    if '-' in i:
      st = i.split('-')
    if i in both:
      arr = []
      for case in cases:
        for s in st:
          try:
            arr.append(morph.parse(s)[0].inflect({case}).word)
          except:
            continue
      arr = [(arr[indx], arr[indx+1]) for indx, x in enumerate(arr) if indx%2 == 0]
      rus_index[i] = [' '.join(x) for x in arr]
    elif i in tcsh:
      arr = []
      for fl in ['ему', 'ий', 'его', 'им', 'ем']:
        arr.append(re.sub('ий', fl, i))
      rus_index[i] = [''.join(x) for x in arr]
    elif i in except_last:
      arr = []
      for case in cases:
        for s in st:
          if s == st[-1]:
            arr.append(s)
            continue
          arr.append(morph.parse(s)[0].inflect({case}).word)
      if len(st) == 3:
        arr = [(arr[indx], arr[indx+1], arr[indx+2]) for indx, x in enumerate(arr) if indx%3 == 0]
      elif len(st) == 2:
        arr = [(arr[indx], arr[indx+1]) for indx, x in enumerate(arr) if indx%2 == 0]
      rus_index[i] = [' '.join(x) for x in arr]
    elif i in last:
        arr = []
        for case in cases:
          try:
            arr.append((st[0], st[1], st[2], morph.parse(st[3])[0].inflect({case}).word))
          except:
            continue
        rus_index[i] = [' '.join(x) for x in arr]
    else:
        arr = []
        for case in cases:
          try:
            arr.append((morph.parse(st[0])[0].inflect({case}).word, st[1]))
          except:
            continue
        rus_index[i] = [' '.join(x) for x in arr]

  rus_index['сыновья дити'].append('сыновей дити')
  rus_index['сыновья калаки'].append('сыновей калаки')

  return rus_index

Разделение санскритизмов на 3 группы: несклоняемые; оканч. на гласную; окан. на согласную

In [3]:
def group_sans(index3, sans, sans_dict2, phrase, rus_index):

  indecl = []
  decl = []
  stem = []
  vows = ['а', 'я', 'ы']
  for i in index3 + sans + sans_dict2:
      if i.endswith('и') or i.endswith('у') or i.endswith('ю') or i.endswith('е') or i.endswith('о'):
          indecl.append(i)
      elif i[-1:] in vows:
          decl.append((i[:-1], i))
      else:
          stem.append(i)
  for i in tr:
    decl.append((i[:-1], i))

  indecl = set(indecl)
  stem = set(stem)

  #exep - список исключений

  preps = ['в', 'без', 'до', 'из', 'к', 'на', 'по', 'о', 'от', 'перед', 'при', 'через', 'с', 'у', 'за', 'над', 'об', 'под', 'про', 'для', 'ко', 'обо', 'ото', 'во', 'безо', 'передо', 'со', 'надо']
  prons = ['меня', 'нем', 'наш', 'ваш', 'она', 'они', 'кто', 'что', 'ними', 'ней', 'наши', 'ваши', 'вашем', 'тот', 'та', 'те', 'той', 'нас', 'ней', 'нам', 'них']
  conjs = ['или', 'но', 'да', 'либо', 'ради', 'нет', 'да']
  advs = ['так', 'туда', 'как', 'тут', 'там', 'сами', 'сам', 'сама']
  exep = preps + prons + conjs + advs

  indecl = [x for x in indecl if x not in exep]

  #словари с начальными буквами для ускоренной обработки
  decl_d = {}
  stem_d = {}

  for i in list(set([x[0][:2] for x in decl])):
    decl_d[i] = []
  for i in list(set([x[:2] for x in stem])):
    stem_d[i] = []

  for i in decl:
    decl_d[i[0][:2]].append(i)
  for i in stem:
    stem_d[i[:2]].append(i)

  #список рубрик с дефисами
  hyph = []

  for i in list(phrase.keys()):
    if re.search('^[а-яА-ЯёЁ]+-[а-яА-ЯёЁ]+$', i):
      hyph.append(re.search('^[а-яА-ЯёЁ]+-[а-яА-ЯёЁ]+$', i).group(0))

  rest = [x for x in list(phrase.keys()) if x not in list(rus_index.keys())]
  hyph.append('тапас-брихадуктха')
  phrase['тапас-брихадуктха'] = 'тапас-брихадуктха'

  return indecl, decl_d, stem_d, exep, hyph, phrase, rest

Поиск санскритизмов в русском тексте

In [4]:
def search(file, rus_index, phrase, rest, hyph, rus_words, indecl, decl_d, stem_d, exep):

  with open(path + file, 'r', encoding = 'utf-8') as r:
    text = r.read()

  reg_space = re.compile('\s{2,}', re.DOTALL)
  reg_punct = ("\s[0-9!\(\),-:;\?\[\]«»'–“„\.•…]+|[0-9!\(\),-:;\?\[\]«»'–“„\.•…]+\s|[0-9!\(\),-:;\?\[\]«»'–“„\.•…]+$|[\(\)«»'“„]+")
  sents = nltk.sent_tokenize(text)

  decl_lemma_sets = {k: set(x[1] for x in v) for k, v in decl_d.items()}
  stem_sets = {k: set(v) for k, v in stem_d.items()}

  found = []
  upper_words = []

  count = 0
  for sent in sents:
    count += 1
    print('Обработано ' + str(count) + ' предложений из ' + str(len(sents)))

    if re.search('[а-яА-Я](-\s-[0-9]+-\s)[а-яА-Я]+', sent):
      br = re.search('[а-яА-Я](-\s-[0-9]+-\s)[а-яА-Я]+', sent).group(1)
      sent = re.sub(br, '', sent)

    text = re.sub(r'\d', '', sent)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[A-Za-z]', '', text)
    text = re.sub(reg_punct, ' ', text)
  
    if re.search(reg_space, text):
      text = re.sub(reg_space, ' ', text)

    sent_low = sent.lower()
    text_low = text.lower()
    for k, vs in rus_index.items():
      if k in sent_low or k in text_low:
        if k in phrase:
          found.append(phrase[k])
        else:
          found.append(k)
      else:
        for v in vs:
          if v in sent_low or v in text_low:
            if k in phrase:
              found.append(phrase[k])
            else:
              found.append(k)

    for i in rest:
      if i in sent_low or i in text_low:
        found.append(phrase[i])

    for i in hyph:
      arr = i.split('-')
      reg = '{}[а-я]+-{}[а-я]+'.format(arr[0][:-2], arr[1][:-2])
      if re.search(reg, sent_low):
        found.append(phrase[i])
      elif re.search(reg, text_low):
        found.append(phrase[i])

    vs = ['а', 'я', 'ю', 'е', 'о', 'ы', 'и', 'у']
    #проверяем каждое слово из текста
    for indx, word in enumerate(text.split()):
        f = False
        if word[0].isupper() == True and indx > 0:
          upper_words.append((word.lower(), indx, text))
        word = word.lower()
        #сохраняем слово, если оно в списке несклоняемых имен
        if word in indecl and word not in rus_words:
            found.append((word, indx, text))
          
        #список склоняемых санскритизмов, заканчивающихся на гласную
        if word[:2] in decl_d:
          if word in decl_lemma_sets[word[:2]]:
              found.append((word, indx, text))
              f = True
          else:
            for i in decl_d[word[:2]]:
                s = i[0]
                if word.startswith(s) and word[-1] in vs and len(word) - len(s) == 1 and word not in exep and word not in rus_words:
                    found.append((i[1], indx, text))
                    f = True
                elif len(word) > 2 and word.startswith(s) and word[-3] in vs and word.endswith('ми') and len(word) - len(s) == 3 and word not in exep and word not in rus_words:
                    found.append((i[1], indx, text))
                    f = True
                elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('м') and len(word) - len(s) == 2 and word not in exep and word not in rus_words:
                    if word[-2:] == 'ам' or word[-2:] == 'ям':
                      if i[1][-1] == 'и' or i[1][-1] == 'ы':
                        found.append((i[1], indx, text))
                        f = True
                    else:
                      found.append((i[1], indx, text))
                      f = True
                elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('в') and len(word) - len(s) == 2 and word not in exep and word not in rus_words:
                    if i[1][-1] == 'и' or i[1][-1] == 'ы':
                      found.append((i[1], indx, text))
                      f = True
                elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('й') and len(word) - len(s) == 2 and word not in exep and word not in rus_words:
                    found.append((i[1], indx, text))
                    f = True
                elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('х') and len(word) - len(s) == 2 and word not in exep and word not in rus_words:
                    if i[1][-1] == 'и' or i[1][-1] == 'ы':
                      found.append((i[1], indx, text))
                      f = True
                    elif word[:-1] == i[1]:
                      found.append((i[1], indx, text))
                      f = True


        if word[:2] in stem_d:
          if word in stem_sets[word[:2]]:
            found.append((word, indx, text))
          else:
            for s in stem_d[word[:2]]:
                if word.startswith(s) and word[-1] != 'ь' and len(word) - len(s) <= 1 and word not in exep and word not in rus_words:
                  if word[-1] in vs and s[-1] in vs:
                    found.append((s, indx, text))
                  elif word[-1] not in vs and s[-1] not in vs:
                    found.append((s, indx, text))
                elif len(word) > 2 and word.startswith(s) and word[-3] in vs and word.endswith('ми') and len(word) - len(s) == 3 and word not in exep and word not in rus_words:
                    found.append((s, indx, text))
                elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('м') and len(word) - len(s) == 2 and word not in exep and word not in rus_words:
                  if word[-2:] == 'ам' or word[-2:] == 'ям':
                    if s[-1] == 'и' or s[-1] == 'ы':
                      found.append((s, indx, text))
                  else:
                    found.append((s, indx, text))
                elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('в') and len(word) - len(s) == 2 and word not in exep and word not in rus_words:
                    found.append((s, indx, text))
                elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('й') and len(word) - len(s) == 2 and word not in exep and word not in rus_words:
                    found.append((s, indx, text))
                elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('х') and len(word) - len(s) == 2 and word not in exep and word not in rus_words:
                    found.append((s, indx, text))

  return found, upper_words

Обработка слов с заглавной буквы

In [5]:
def capital_search(upper_words, sans, index3, tr, exep):
  found = []
  vs = ['а', 'я', 'ю', 'е', 'о', 'ы', 'и', 'у']
  pool = sans + index3 + tr
  for i in upper_words:
    word = i[0]
    for s in pool:
        if word.startswith(s) and word[-1] != 'ь' and len(word) - len(s) <= 1 and word not in exep:
            found.append((s, i[1], i[2]))
        elif len(word) > 2 and word.startswith(s) and word[-3] in vs and word.endswith('ми') and len(word) - len(s) == 3 and word not in exep:
            #if s[-1] == 'и' or s[-1] == 'ы':
            found.append((s, i[1], i[2]))
        elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('м') and len(word) - len(s) == 2 and word not in exep:
          if word[-2:] == 'ам' or word[-2:] == 'ям':
            if s[-1] == 'и' or s[-1] == 'ы':
              found.append((s, i[1], i[2]))
          else:
            found.append((s, i[1], i[2]))
        elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('в') and len(word) - len(s) == 2 and word not in exep:
            if s[-1] == 'и' or s[-1] == 'ы':
              found.append((s, i[1], i[2]))
        elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('й') and len(word) - len(s) == 2 and word not in exep:
            found.append((s, i[1], i[2]))
        elif len(word) > 1 and word.startswith(s) and word[-2] in vs and word.endswith('х') and len(word) - len(s) == 2 and word not in exep:
            if s[-1] == 'и' or s[-1] == 'ы':
              found.append((s, i[1], i[2]))

  #добавляю невстречающиеся в тексте слова
  to_append = {}
  with open(path + 'append if found.txt', 'r', encoding='utf-8') as r:
    for i in r:
      s = i.split(': ')
      to_append[s[0]] = re.sub('\n', '', s[1])

  for i in found:
    if isinstance(i, str) == True:
      if i in to_append:
        found.append(to_append[i])
    else:
      if i[0] in to_append:
        found.append(to_append[i[0]])

  ffound = []
  for line in found:
    if line[0] not in opts:
      ffound.append(line)
    else:
      opt = opts[line[0]]
      f = False
      for i in opt:
        sp = i[0].split()
        for s in sp:
          if s[:-1] in line[2].lower():
            ffound.append((i[1], line[1], line[2]))
            f = True
      if f == False:
        ffound.append(line)
  return ffound

Приведение рубрик к рубрикам из бумажного указателя

In [6]:
def index_transform(found, upper_words, rusforms, sans, index3, tr, exep):

  found = [x for x in found if x[0].endswith('ам') == False]

  #добавляю невстречающиеся в тексте слова
  to_append = {}
  with open(path + 'append if found.txt', 'r', encoding='utf-8') as r:
    for i in r:
      s = i.split(': ')
      to_append[s[0]] = re.sub('\n', '', s[1])

  for i in found:
    if isinstance(i, str) == True:
      if i in to_append:
        found.append(to_append[i])
    else:
      if i[0] in to_append:
        found.append(to_append[i[0]])


  #переименовываю на эпитеты
  ffound = []
  for line in found:
    if isinstance(line, tuple):
      if line[0] not in opts:
        ffound.append(line)
      else:
        opt = opts[line[0]]
        f = False
        for i in opt:
          sp = i[0].split()
          for s in sp:
            if s[:-1] in line[2].lower():
              ffound.append((i[1], line[1], line[2]))
              f = True
        if f == False:
          ffound.append((line[0], line[1], line[2]))
    else:
      ffound.append([line])

  sans = [x for x in sans if len(x) > 1]

  up_words = list(set([x for x in upper_words if x[0] not in [x[0] for x in found]]))
  postfound = capital_search(up_words, sans, index3, tr, exep)
  postfound = [x for x in postfound if str(x[0]).endswith('ам') == False]
  ffound = ffound + postfound

  res = []
  for i in ffound:
    if i[0] in repl:
      if len(i) > 1:
        res.append((repl[i[0]], i[1], i[2]))
      else:
        res.append([repl[i[0]]])
    else:
      res.append(i)

  res = [x for x in res if x[0] not in rusforms]

  return res

Отметение вариантов лемм по окончанию

In [7]:
def unite1(res):

  options = {}
  clean = []

  for i in res:
    if len(i) == 3:
      k = str(i[1])
      if (k, i[2]) not in options:
        options[(str(i[1]), i[2])] = [i[0]]
      else:
        options[(str(i[1]), i[2])].append(i[0])
    else:
      clean.append(i[0])

  opts = {}
  for k, vs in options.items():
    if len(set(vs)) > 1:
      f = False
      for v in vs:
        if re.search('[–\,\(\)\.-]', v):
          clean.append([v][0])
          f = True
          break
      if f == False:
        opts[k] = list(set(vs))
    else:
      clean.append(list(set(vs))[0])

  return opts, clean

In [8]:
def unite2():

  # са/си, ла/ли, ра/ри, на/ни, за/зи, фа/фи, ва/ви, па/пи, да/ди, ма/ми, та/ти, ба/би.
  # буквосочетания выше выделены искусственно (из всех возможных согласных на конце слова перед -и или -а)

  #шардула/шардули; ракшаса/ракшаси; 

  solid = ['с', 'л', 'р', 'н', 'з', 'ф', 'в', 'п', 'д', 'м', 'т', 'б']
  vos = ['а', 'я', 'ю', 'е', 'о', 'ы', 'и']

  solid_opts = {}
  to_set = {}
  for_dpavlov = {}

  opts, clean = unite1(res)

  for k, vs in opts.items():
    for indx, i in enumerate(k[1].split()):
      if str(indx) == k[0]:
        for v in vs:
          #ракшасов
          #на -в могут заканчиваться слова только во мн. числе, поэтому лемма должна заканчиваться на -и или -ы, т.е. стоять во мн. числе
          if i[-1] == 'в':
            if len([m for m in vs if m.endswith('ы') or m.endswith('и')]) > 0:
              if v[-1] == 'ы' or v[-1] == 'и':
                if k not in solid_opts:
                  solid_opts[k] = [v]
                else:
                  solid_opts[k].append(v)
          elif i.endswith('ой'):
            # if len([m for m in vs if m.endswith('а')]) > 0:
            if v[-1] != 'ы' and v[-1] != 'и':
              if k not in solid_opts:
                solid_opts[k] = [v]
              else:
                solid_opts[k].append(v)
          
          #читрами
          #на -ми могут заканчиваться слова только во мн. числе, поэтому лемма должна заканчиваться на -и или -ы, т.е. стоять во мн. числе
          elif i.endswith('ми'):
            if len([m for m in vs if m.endswith('ы') or m.endswith('и')]) > 0:
              if v[-1] == 'ы' or v[-1] == 'и':
                if k not in solid_opts:
                  solid_opts[k] = [v]
                else:
                  solid_opts[k].append(v)

          #в тексте слово "шастрах", варианты леммы - "шастра", "шастры"
          elif i[-1] == 'х':
            if len([m for m in vs if m.endswith('ы') or m.endswith('и')]) > 0:
              #если лемма заканчивается на "ы" или "и", то выбираем её (так как слово в тексте заканчивается на "х", соответственно, стоит во мн. числе)
              #и лемма должна быть во мн. числе
              if v[-1] == 'ы' or v[-1] == 'и':
                #сохраняется лемма, предложение и индекс ее словоформы в предложении для дальнейшей обработки
                if k not in solid_opts:
                  solid_opts[k] = [v]
                else:
                  solid_opts[k].append(v)

          #шардулу
          #на -у могут заканчиваться слова только во ед. числе, поэтому лемма не должна заканчиваться на -и или -ы, т.е. стоять в ед. числе
          elif i[-1] in ['у', 'ю', 'е', 'о']:
            if v[-1] != 'ы' and v[-1] != 'и':
              if k not in solid_opts:
                solid_opts[k] = [v]
              else:
                solid_opts[k].append(v)
          
          #шардулам, валакхильям
          #на -ам и -ям могут заканчиваться слова только во мн. числе, поэтому лемма должна заканчиваться на -и или -ы, т.е. стоять во мн. числе
          elif i.endswith('ам') or i.endswith('ям'):
            if v[-1] == 'ы' or v[-1] == 'и':
              if k not in solid_opts:
                solid_opts[k] = [v]
              else:
                solid_opts[k].append(v)
          elif i.endswith('ом') or i.endswith('ем'):
            if v[-1] != 'ы' and v[-1] != 'и':
              if k not in solid_opts:
                solid_opts[k] = [v]
              else:
                solid_opts[k].append(v)

          # слово "кадали", варианты - "кадала", "кадали".
          # слово "кадала", варианты - "кадала", "кадали" - искусственный пример
          elif i[-1] in vos and i[-2] in solid:
            # оставляем "кадала", когда слово "кадала"
            if i[-1] != 'и' and v[-1] != 'и':
              if i[-1] != 'х' and i[-1] != 'в' and not i.endswith('ам') and not i.endswith('ям') and not i.endswith('ми') and v[-1] != 'ы':
                if k not in solid_opts:
                  solid_opts[k] = [v]
                else:
                  solid_opts[k].append(v)
            #оставляем "кадали", когда слово "кадали"
            elif i[-1] == 'и' and v[-1] == 'и':
              if k not in solid_opts:
                solid_opts[k] = [v]
              else:
                solid_opts[k].append(v)
          else:
            for_dpavlov[k] = vs
  return clean, for_dpavlov

Разбор deeppavlov-а

In [9]:
#получение всех словоформ леммы в тексте

_wf_token_cache = {}


def get_wordforms(word, file):

  key = path + file
  toks = _wf_token_cache.get(key)
  if toks is None:
    with open(path + file, 'r', encoding = 'utf-8') as r:
      text = r.read()

    reg_space = re.compile('\s{2,}', re.DOTALL)
    reg_punct = ("\s[0-9!\(\),-:;\?\[\]«»'–“„\.•…]+|[0-9!\(\),-:;\?\[\]«»'–“„\.•…]+\s|[0-9!\(\),-:;\?\[\]«»'–“„\.•…]+$|[\(\)«»'“„]+")
    if re.search('[а-яА-Я](-\s-[0-9]+-\s)[а-яА-Я]+', text):
      br = re.search('[а-яА-Я](-\s-[0-9]+-\s)[а-яА-Я]+', text).group(1)
      text = re.sub(br, '', text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[A-Za-z]', '', text)
    text = re.sub(reg_punct, ' ', text)
    if re.search(reg_space, text):
      text = re.sub(reg_space, ' ', text)
    text = text.lower()

    toks = text.split()
    _wf_token_cache[key] = toks

  forms = []
  for i in toks:
    if i == word:
      forms.append(i)
    elif word.startswith(i[:-1]) and len(word) == len(i) and len(i) > 3:
      forms.append(i)
    elif word.startswith(i[:-1]) and len(i) - len(word) == 1 and len(i) > 3:
      forms.append(i)
    elif word.startswith(i[:-2]) and len(i) - len(word) < 3 and i[-1] in ['в', 'й', 'х', 'м'] and len(i) > 3 and word[-2] == i[-3]:
      forms.append(i)
    elif word.startswith(i[:-2]) and len(i) - len(word) < 3 and i[-1] in ['в', 'й', 'х', 'м'] and len(i) > 3 and word[-1] == i[-3]:
      forms.append(i)
    elif word.startswith(i[:-3]) and len(i) - len(word) < 4 and i.endswith('ми') and len(i) > 4:
      forms.append(i)
  
  return forms

Отметение оставшихся вариантов лемм с помощью разбора deeppavlov

In [10]:
def depppavlov_proc(file, for_dpavlov, clean):

  # чтение файла с синт. разбором предложений текста

  d = {}

  with open(path + 'deeppavlov_{}'.format(file), 'r', encoding = 'utf-8') as r:
    for i in r:
      if re.search('\((.*)\,(\[.*])', i):
        sent = re.search('\((.*)\,(\[.*])', i).group(1)
        synt = re.search('\((.*)\,(\[.*])', i).group(2)
        d[sent] = synt

  clean_case = []

  vs = ['а', 'я', 'ю', 'е', 'о', 'ы', 'и']

  for k, v in for_dpavlov.items():

    text = k[1]
    for indx, i in enumerate(text.split()):
      if str(indx) == k[0]:
        wordform = i.lower()

    if text in d:
      elem = d[text]

      number = [i for i in range(int(k[0])-3, int(k[0])+4)]
      arr = []
      for i in number:
        if number[-1] != i:
          st = '(' + str(i) + ')|'
          arr.append(st)
        else:
          st = '(' + str(i) + ')'
          arr.append(st)

      st = ''.join(arr)
      s = '(' + st + ')'

      if re.search(r'({})\\t{}[а-яё]*[\\ta-z=|_]+case=nom[\\ta-z=|_]+\\t{}'.format(wordform, wordform[:-1], s), elem.lower()):
        if wordform in v:
          clean_case.append((wordform, k[0], k[1]))
          # print((wordform, k[0], k[1]))
          continue
        else:
          clean_case.append((wordform, k[0], k[1]))


      elif re.search(r'({})\\t{}[а-яё]*[\\ta-z=|_]+case=(gen|dat|acc|ins|loc)[\\ta-z=|_]+\\t{}'.format(wordform, wordform[:-1], s), elem.lower()):
        check = False
        for lem in v:
          #Урваши
          forms = get_wordforms(lem, file)
          #если слово в других формах в тексте не встречается, то принимаем его за лемму
          if bool(forms.count(forms[0]) == len(forms)) and lem == forms[0]:
            clean_case.append((lem, k[0], k[1]))
            check == True
        
        #но если слово встречается в других формах в тексте..
        #словоформа "ракшасу", варианты леммы - ракшас, ракшаси. 
        if check == False:
          if len(v) == 2:
            if v[0].endswith('и') == True and v[1].endswith('и') == False:
              clean_case.append((v[1], k[0], k[1]))
            elif v[0].endswith('и') == False and v[1].endswith('и') == True:
              clean_case.append((v[0], k[0], k[1]))
            else:
              for lem in v:
                for i in [x[0] for x in clean_case]:
                  if lem[-1] in vs:
                    if i.startswith(lem[:-1]):
                      clean_case.append((i, k[0], k[1]))
                      break
      else:
        clean_case.append((wordform, k[0], k[1]))
    else:
      print('text not in d')

  for x in clean_case:
    clean.append(x[0])

  return clean

Объединение рубрик указателя

In [11]:
def index_unite(clean):
  lexemes = []

  uniq = list(set(clean))
  word_of = {l: re.match('[а-яА-Я]+', l).group(0) for l in uniq if re.match('[а-яА-Я]+', l)}

  for line in uniq:
    if line in word_of:
      word = word_of[line]
      check = False
      if word in rus_words:
        lexemes.append(line)
        continue

      wordforms = []
      wordforms.append(word)

      if word[-1] != 'ю' and word[-1] != 'у':
        for l in uniq:
          word2 = word_of.get(l)
          if word2 is None:
            word2 = re.match('[а-яА-Я]+', l).group(0)
          if word != word2:

            if len(word) > len(word2) and len(word) - len(word2) <= 1:
              if word2.startswith(word[:-1]) or word.startswith(word2[:-1]):
                if word[-1] in vs and word2[-1] not in vs:
                  if word[:-1] == word2:
                    check = True
                    wordforms.append(word2)
                    # print((word, word2))
                elif word2[-1] in vs and word[-1] not in vs:
                  if word2[:-1] == word:
                    check = True
                    wordforms.append(word2)
                    # print((word, word2))
                elif word[-1] in vs and word2[-1] in vs:
                  if word[:-1] == word2[:-1]:
                    check = True
                    wordforms.append(word2)
                    # print((word, word2))

            elif len(word2) > len(word) and len(word2) - len(word) <= 1:
              if word2.startswith(word[:-1]) or word.startswith(word2[:-1]):
                if word[-1] in vs and word2[-1] not in vs:
                  if word[:-1] == word2:
                    check = True
                    wordforms.append(word2)
                    # print((word, word2))
                elif word2[-1] in vs and word[-1] not in vs:
                  if word2[:-1] == word:
                    check = True
                    wordforms.append(word2)
                    # print((word, word2))
                elif word[-1] in vs and word2[-1] in vs:
                  if word[:-1] == word2[:-1]:
                    check = True
                    wordforms.append(word2)
                    # print((word, word2))

            elif len(word2) == len(word):
              if word2.startswith(word[:-1]) or word.startswith(word2[:-1]):
                if word[-1] in vs and word2[-1] not in vs:
                  if word[:-1] == word2:
                    check = True
                    wordforms.append(word2)
                    # print((word, word2))
                elif word2[-1] in vs and word[-1] not in vs:
                  if word2[:-1] == word:
                    check = True
                    wordforms.append(word2)
                    # print((word, word2))
                elif word[-1] in vs and word2[-1] in vs:
                  if word[:-1] == word2[:-1]:
                    check = True
                    wordforms.append(word2)
                    # print((word, word2))

      # if len(wordforms) > 1:
      if check == True:
        lexemes.append(wordforms)
      else:
        lexemes.append(line)
  return lexemes

Запись получившегося автоматического указателя в файл

In [12]:
def get_index(lexemes, file):
  unit_clean = list(set([x for x in lexemes if isinstance(x, str)]))

  united = []
  for forms in lexemes:
    if [x for x in forms if x[-1] not in vs]:
      united.append([x for x in forms if x[-1] not in vs][0])
    elif [x for x in forms if x[-1] not in ['и', 'ы']]:
      united.append([x for x in forms if x[-1] not in ['и', 'ы']][0])

  for i in unit_clean:
    united.append(i)

  united = [x for x in united if len(x) > 1]
  united = [x for x in united if x not in ['эха', 'свита', 'рук', 'матери', 'правая']]

  with open(path + 'automated_index_{}'.format(file), 'w', encoding='utf-8') as m:
    for i in sorted(list(set(united))):
      m.write(i)
      m.write('\n')
  return united

Нахождение всех словоформ рубрик из указателя в тексте

In [13]:
def get_index_forms(united, file, phrase, repl, rus_words, rus_index):

  united_forms = {}
  for line in list(set(united)):
    if re.match('[а-яА-Я]+', line):
      lems = []
      word = re.match('[а-яА-Я]+', line).group(0)
      lems.append(word)
      #чтобы к джатаю(с) находилось джатаюс, помимо джатаю
      if re.match('[а-яА-Я]+(\([а-яА-Я]{1,2}\))', line):
        full_form = word + re.match('[а-яА-Я]+\(([а-яА-Я]{1,2})\)', line).group(1)
        lems.append(full_form)

      
      for word in lems:
        if line not in [x for x in list(phrase.values())] and line not in [x for x in list(repl.values())]:
          forms = get_wordforms(word, file)
          forms = [x for x in forms if x not in rus_words]
          forms.append(word)
          if line not in united_forms:
            united_forms[line] = list(set(forms))
          else:
            for i in list(set(forms)):
              united_forms[line].append(i)
        else:
          check = False
          for k, v in phrase.items():
            if line == v:
              cl = k
              check = True
              break
          if check == False:
            for k, v in repl.items():
              if line == v:
                cl = k
                break
          for k, v in rus_index.items():
            if cl == k:
              w = v
              w.append(k)
              if line not in united_forms:
                united_forms[line] = list(set(w))
              else:
                for i in list(set(w)):
                  united_forms[line].append(i)
          if cl not in rus_index:
            forms = get_wordforms(word, file)
            forms = [x for x in forms if x not in rus_words]
            forms.append(word)
            if line not in united_forms:
              united_forms[line] = list(set(forms))
            else:
              for i in list(set(forms)):
                united_forms[line].append(i)

  with open(path + 'automated_index_forms_{}'.format(file), 'w', encoding='utf-8') as m:
    for k, v in united_forms.items():
      m.write(k)
      m.write(' : ')
      m.write(str(v))
      m.write('\n')